# Ejercicio 5 (OPCIONAL) — Actualización de la "tabla de escrituras" 

En nuestro diseño (Tarea 1) las tablas clave son:
- `hall_of_fame_by_country`  
  `PRIMARY KEY ((country, dungeon_id), time_minutes, email)`  
  y además `user_name`, `date`, `dungeon_name STATIC`
- `dungeons_by_country`  
  `PRIMARY KEY ((country), dungeon_id)`
- `user_statistics_by_dungeon`  
  `PRIMARY KEY ((email, dungeon_id), time_minutes, date)`
- `top_horde_by_event`  
  `PRIMARY KEY ((country, event_id), n_killed, email)`  
  y además `user_name`  
  Con un índice secundario sobre `email` (creado en la tarea 4) para poder buscar el `n_killed` actual de un jugador.

Por tanto, las escrituras del enunciado original **no son suficientes** para escribir directamente en estas tablas sin hacer lecturas extra.


## Escrituras actualizadas 

### When: User finish dungeon
**Antes (enunciado):**
- `dungeon_id: int`
- `email: str`
- `time_minutes: float`
- `date: str` (ISO 8601)

**Después (adaptado a nuestras tablas Cassandra):**
- `country: str`  
  Necesario para escribir en `hall_of_fame_by_country`, cuya partición es `(country, dungeon_id)`, y en `dungeons_by_country`, cuya partición es `(country)`.
- `dungeon_id: int`
- `dungeon_name: str`  
  Se devuelve en el "Hall of Fame" y se guarda como `STATIC` en `hall_of_fame_by_country`.  
  Sin este campo, el servidor tendría que consultar la tabla relacional `Dungeon` (o una tabla Cassandra equivalente).
- `email: str` (identificador del usuario en nuestro diseño)
- `user_name: str`  
  Se devuelve en el "Hall of Fame" (Top 5), y se guarda denormalizado para evitar consultar `WebUser`.
- `time_minutes: int/float`
- `date: str` (ISO 8601)

Aunque exista la tabla `dungeons_by_country`, esta solo guarda ids; no aporta `dungeon_name`, por eso es útil que el evento lo envíe directamente.



### When: User kills monster during Horde event
**Antes (enunciado):**
- `event_id: int`
- `email: str`
- `monster_id: int`

**Después (adaptado a nuestras tablas Cassandra):**
- `country: str`  
  Necesario porque `top_horde_by_event` particiona por `(country, event_id)` y el ranking de horda es local por país.
- `event_id: int`
- `email: str` (identificador del usuario)
- `user_name: str`  
  Se devuelve en la lectura del Top Horde, y se guarda denormalizado para evitar leer `WebUser`.
- `n_killed: int` 
  El número total de monstruos matados por el usuario en ese evento.  
  Se puede calcular a partir del número actual (consultando el índice secundario sobre `email`). No usamos `monster_id` porque no es necesario para el ranking, y así evitamos lecturas extra para consultar el tipo de monstruo (además de que dentro de Cassandra, las operaciones de agrupación y conteo son poco eficientes).


### Justificación de nuestro diseño
1. **Los leaderboards son locales por país**, y nuestro diseño Cassandra usa `country` como parte de la **clave de partición** en `hall_of_fame_by_country` y `top_horde_by_event`. Si `country` no llega en la escritura, no se puede escribir en la partición correcta sin consultar la tabla de usuarios.
2. En Cassandra **no hay JOINs**, y queremos que cada evento de escritura se traduzca en **escrituras directas** sobre las tablas denormalizadas. Si faltan `country`, `user_name` o `dungeon_name`, el backend tendría que hacer lecturas previas para obtenerlos (p.ej., mirar `WebUser.country/userName` o `Dungeon.name`).
3. Guardamos `user_name` y `dungeon_name` denormalizados porque **aparecen en los outputs** de las lecturas. Así, cuando el cliente pide el leaderboard, Cassandra ya tiene toda la información lista y no necesita combinar tablas.
4. `dungeon_name` se almacena como **STATIC** en `hall_of_fame_by_country` para no repetir el mismo nombre en cada fila del Top-5: queda guardado una sola vez por partición `(country, dungeon_id)`.
5. Para las Hordas, el enunciado dice que la latencia durante gameplay es crítica. Evitar lecturas adicionales por cada kill es esencial; con `country` y `user_name` en el evento, la actualización del ranking es inmediata.


# CQL de escrituras usando el nuevo cuerpo de la petición 

## When: User finish dungeon

Este evento dispara escrituras en **tres tablas**: `hall_of_fame_by_country`, `user_statistics_by_dungeon` y `dungeons_by_country`.

### Insert en `hall_of_fame_by_country`
```sql
INSERT INTO hall_of_fame_by_country (
  country, dungeon_id, time_minutes, email, user_name, date, dungeon_name
) VALUES (
  :country, :dungeon_id, :time_minutes, :email, :user_name, :date, :dungeon_name
);
```

### Insert en `user_statistics_by_dungeon`
```sql
INSERT INTO user_statistics_by_dungeon (
  email, dungeon_id, time_minutes, date
) VALUES (
  :email, :dungeon_id, :time_minutes, :date
);
```

### Insert en `dungeons_by_country`
Para asegurar que la combinación país-mazmorra queda registrada. Si ya existe, Cassandra simplemente sobrescribe con los mismos valores (operación idempotente).
```sql
INSERT INTO dungeons_by_country (
  country, dungeon_id
) VALUES (
  :country, :dungeon_id
);
```

`date` es TIMESTAMP en nuestras tablas, así que el backend debe convertir ISO 8601 a timestamp (o enviarlo ya como timestamp compatible).

## When: User kills monster during Horde event

### Importante sobre nuestra tabla top_horde_by_event
* Esta tabla está ordenada por `n_killed DESC` y `n_killed` forma parte de la clave primaria.
Eso significa que cuando cambie `n_killed`, no podemos hacer UPDATE de ese valor en la misma fila, ya que en Cassandra cambiar un componente de la clave equivale a insertar una nueva fila con otra clave (y la antigua seguiría existiendo si no la borrásemos).

### Enfoque elegido: el backend consulta el `n_killed` actual mediante el índice secundario y reinserta

El flujo es el siguiente:

**Paso 1** — Obtener el `n_killed` actual gracias al índice secundario sobre `email` creado en la tarea 4:
```sql
SELECT n_killed FROM top_horde_by_event
WHERE country = :country AND event_id = :event_id AND email = :email;
```
Esta consulta utiliza el índice secundario sobre `email` para poder filtrar por una columna de clustering sin necesidad de especificar `n_killed`.

**Paso 2** — Calcular `new_n_killed = old_n_killed + 1` (en el backend).

**Paso 3** — Borrar la fila antigua e insertar la nueva en un BATCH:
```sql
BEGIN BATCH
  DELETE FROM top_horde_by_event
  WHERE country = :country AND event_id = :event_id AND n_killed = :old_n_killed AND email = :email;

  INSERT INTO top_horde_by_event (country, event_id, n_killed, user_name, email)
  VALUES (:country, :event_id, :new_n_killed, :user_name, :email);
APPLY BATCH;
```

La lectura del Top-K se obtiene directamente con:
```sql
SELECT ... FROM top_horde_by_event WHERE country = ? AND event_id = ? LIMIT K;
```
(gracias al orden `n_killed DESC`).

### ¿Por qué no usamos tablas COUNTER?
Se podría pensar en crear una tabla con columnas de tipo `COUNTER` para contar kills por usuario. Sin embargo, en Cassandra las tablas COUNTER tienen restricciones importantes: no pueden mezclar columnas COUNTER con columnas normales no-COUNTER (salvo las que forman la clave primaria), y no soportan INSERT ni DELETE, solo UPDATE con incrementos. Esto haría imposible mantener `user_name` en la misma tabla e imposibilitaría el patrón DELETE + INSERT que necesitamos en `top_horde_by_event`. Por estas razones descartamos este enfoque.

### Ventajas de este enfoque
* **Lectura del Top-K directa**: `SELECT ... FROM top_horde_by_event WHERE country = ? AND event_id = ? LIMIT K;` devuelve los resultados ya ordenados por `n_killed DESC`.
* **Consistencia aceptable**: durante el breve intervalo entre el DELETE y el INSERT, el ranking puede "bailar" ligeramente, lo cual el enunciado considera aceptable e incluso motivador para los jugadores.
* **BATCH**: agrupamos el DELETE e INSERT en un BATCH para reducir la ventana de inconsistencia.
* **Índice secundario sobre `email`**: permite obtener el `n_killed` actual de un jugador sin necesidad de tablas auxiliares adicionales. Aunque los índices secundarios en Cassandra generan queries distribuidas, en el contexto de la Horda (donde la consistencia no es crítica y usamos `CONSISTENCY ONE`) es un compromiso aceptable para simplificar el modelo de datos.